# Debye scattering equation (1/2)
This notebook computes the scattered intensity from arbitrary atomic ensembles using the Debye scattering equation:
$$ I \left( Q \right) = \sum_{i=1}^{N}\sum_{j=1}^{N}f_i(Q) f_j(Q) {sin \left(Q r_{ij} \right) \over {Q r_{ij}}}$$
The result is "powder diffraction"-like pattern. The first step of the computation consist in calculating all atomic pair distances $r_{ij}$. The second step in the summation over all values of $r_{ij}$.
For a monoatomic crystal the equation can be written:
$$ I \left( Q \right) = \left| f(Q) \right|^2  \left[ N + 2 \sum_{i=1}^{N}\sum_{j>i}^{N}{sin \left(Q r_{ij} \right) \over {Q r_{ij}}}  \right] $$
For a multi-element crystal this equation simply has to be evaluated for all homo-atomic pairs and all hetero-atomic pairs in the crystal.

Load external libraries.

In [1]:
import math
import numpy as np
import numexpr as ne
import numba as nb
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
import os
n_cpu = os.cpu_count()
from periodictable import cromermann as db

Now we load the atomic coordinates and the coefficients required for the calculation of the atomic scattering factor. The first 11 elements of the coeff array are the coefficients given by [Waasmaier & Kirfel](https://doi.org/10.1107/S0108767394013292) for each element of the periodic table. The last element is the absorption/anomalous dispersion coefficient as given in the [Brennan-Cowan database](https://doi.org/10.1063/1.1142625).

In [2]:
# Atomic coordinates within cluster
coords = np.loadtxt("Au.xyz")
print(coords)
# Atomic scattering factor coefficients
coeff = np.array([16.777390,19.317156,32.979683,5.595453,10.576854,-6.279078,0.122737,
                  8.621570,1.256902,38.008820,0.000601, -4.014837862516359+7.717066375953113j])
# Define 2theta computation range
tth = np.arange(10, 180, 0.2)*np.pi/180
# X-ray wavelength
wl = 1.5406
# Step (in Angstroms) to compute the distance histogram
step=0.0005

[[ 2.035 10.175 20.35 ]
 [ 2.035 10.175 24.42 ]
 [ 2.035 10.175 28.49 ]
 ...
 [48.84  36.63  20.35 ]
 [48.84  36.63  24.42 ]
 [48.84  36.63  28.49 ]]


Global variables computed from the above

In [3]:
N, dim = np.shape(coords) #Extract the number of atoms and the number of dimensions
Q = 4*np.pi*np.sin(tth/2) / wl #Compute the scatttering vector range

In [4]:
#calculate atomic scattering factor f0 using periodictable
def asf(atom_type,Q):
    f0=np.zeros(len(Q))
    f0=db.fxrayatq(atom_type,Q)
    return f0
f0=asf('Au',Q)
print(f0)

[77.66182598 77.6111298  77.5595872  77.50720668 77.45399684 77.39996638
 77.34512407 77.2894788  77.23303951 77.17581522 77.11781505 77.05904817
 76.99952382 76.93925131 76.87824001 76.81649935 76.75403881 76.69086792
 76.62699627 76.56243346 76.49718918 76.43127311 76.36469499 76.29746458
 76.22959167 76.16108607 76.09195762 76.02221616 75.95187156 75.88093368
 75.8094124  75.73731761 75.66465918 75.59144699 75.51769092 75.44340083
 75.36858657 75.29325797 75.21742485 75.14109702 75.06428423 74.98699625
 74.90924278 74.83103352 74.7523781  74.67328615 74.59376724 74.51383089
 74.4334866  74.3527438  74.27161188 74.19010018 74.10821799 74.02597451
 73.94337894 73.86044037 73.77716784 73.69357035 73.6096568  73.52543603
 73.44091684 73.35610792 73.2710179  73.18565533 73.10002871 73.01414643
 72.92801682 72.84164811 72.75504847 72.66822598 72.58118862 72.49394431
 72.40650087 72.31886603 72.23104744 72.14305265 72.05488913 71.96656425
 71.87808529 71.78945946 71.70069384 71.61179544 71

# Part 1: computing pairwise distances

## Pure python implementation

In [5]:
def r_ij_python(coords):
    r = [0 for x in range(int((N*N-N)/2))]
    l = 0
    for i in range(N):
        for j in range(i+1,N):
            tmp = 0
            for k in range(dim):
                tmp += (coords[i,k]-coords[j,k])**2
            r[l] = tmp**0.5
            l += 1
    return np.array(r)

%time r_ij_py = r_ij_python(coords)

CPU times: user 15.5 s, sys: 151 ms, total: 15.7 s
Wall time: 15.7 s


## SciPy implementation 

In [6]:
r_ij_sp = pdist(coords, metric='euclidean')
%timeit pdist(coords, metric='euclidean')
print("Is the result correct:",np.allclose(r_ij_py, r_ij_sp))

27.9 ms ± 247 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Is the result correct: True


## Numba implementation

In [7]:
@nb.jit(nb.float64[:](nb.float64[:,:]), nopython=True, fastmath=True, parallel=True)
def r_ij_numba(coords):
    r = np.zeros(int((N*N-N)/2), dtype=nb.float64)
    for i in nb.prange(N):
        for j in range(i+1,N):
            l = int(i * (N - 1) - i * (i + 1) / 2 + j - 1)
            tmp = 0.0
            for k in range(dim):
                tmp += (coords[i,k]-coords[j,k])**2
            r[l] = tmp**0.5
    return r

%timeit r_ij_numba(coords)
r_ij_nb = r_ij_numba(coords)
print("Is the result correct:",np.allclose(r_ij_py, r_ij_nb))

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


14.3 ms ± 1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
Is the result correct: True


# Part 2: perform the summation over $r_{ij}$

First compute the atomic scattering factor

In [8]:
def asf(th,wl,coeff):
    n = 0
    sum = coeff[5]
    while n <= 4:
        sum = sum + coeff[n] * np.exp ( -coeff[n+6] * ( np.sin(th) / wl )**2)
        n = n+1
    return sum + coeff[-1]
f_at = asf(tth/2, wl, coeff)

## Python implementation

In [ ]:
def Debye_python(Q,r,N,f_at):
    res = [0 for i in range(int(len(Q)))]
    for i_Q, v_Q in enumerate(Q):
        tmp = 0.0
        for v_r in r:
            tmp += math.sin(v_Q*v_r)/(v_Q*v_r)
        res[i_Q] = (N + 2*tmp)*abs(f_at[i_Q])**2
    return np.array(res)
%time ref_intensity = Debye_python(Q,r_ij_nb,N,f_at)

In [ ]:
np.savetxt("ref_intensity.txt", ref_intensity, fmt='%10.8f') #save the file for use with histogram notebook
%matplotlib notebook
plt.semilogy(tth*180/np.pi, ref_intensity)
plt.show()

## Numpy implementation

In [ ]:
def Debye_numpy(Q,r,N,f_at):
    r = r[np.newaxis, :]
    Q = Q[:,np.newaxis]
    res = abs(f_at)**2 * (N + 2*((np.sin(Q*r)/(Q*r)).sum(axis=1)))
    return res

%timeit Debye_numpy(Q,r_ij_nb,N,f_at)
intensity_np = Debye_numpy(Q,r_ij_nb,N,f_at)
print("Error:", abs(intensity_np-ref_intensity).max()/ref_intensity.max())

## NumExpr  implementation

In [ ]:
def Debye_numexpr(Q,r,N,f_at):
    r = r[np.newaxis, :]
    Q = Q[:,np.newaxis]
    ne.set_num_threads(n_cpu)
    res = ne.evaluate("2*sin(Q*r)/(Q*r)") #This must be evaluated before the sum to benefit from multi-threading
    res = ne.evaluate("sum((res), axis=1)")
    res = ne.evaluate("(N + res)*(real(f_at)**2 + imag(f_at)**2)") 
    return res

%timeit Debye_numexpr(Q,r_ij_nb,N,f_at)
intensity_ne = Debye_numexpr(Q,r_ij_nb,N,f_at)
print("Error:", abs(intensity_ne-ref_intensity).max()/ref_intensity.max())

## Numba implementation

In [ ]:
@nb.jit(nb.float64[:](nb.float64[:], nb.float64[:], nb.int64, nb.complex128[:]),
        nopython=True, fastmath=True, parallel=True)
def Debye_numba(Q,r,N,f_at):
    res = np.zeros(int(len(Q)), dtype = nb.float64)
    for i_Q in nb.prange(len(Q)):
        tmp = 0.0
        for i_r in range(len(r)): #"for v_r in r" doens't work (probably bc r is float)
            tmp += math.sin(Q[i_Q]*r[i_r])/(Q[i_Q]*r[i_r])
        res[i_Q] = (N + 2*tmp)*abs(f_at[i_Q])**2
    return res

%timeit Debye_numba(Q,r_ij_nb,N,f_at)
intensity_nb = Debye_numba(Q,r_ij_nb,N,f_at)
print("Error:", abs(intensity_nb-ref_intensity).max()/ref_intensity.max())

## Pythran implementation 

In [ ]:
%reload_ext pythran.magic

In [ ]:
%%pythran -fopenmp

#pythran export Debye_pythran(float[], float[], int, complex[])
import numpy as np

def Debye_pythran(Q,r,N,f_at):
    res = np.zeros(int(len(Q)), dtype = np.float64)
    "omp parallel for"
    for i_Q in range(len(Q)):
        tmp = 0.0
        for i_r in range(len(r)):
            tmp += np.sin(Q[i_Q]*r[i_r])/(Q[i_Q]*r[i_r])
        res[i_Q] = (N + 2*tmp)*abs(f_at[i_Q])**2
    return res

In [ ]:
%timeit Debye_pythran(Q,r_ij_pt,N,f_at)
intensity_pt = Debye_pythran(Q,r_ij_pt,N,f_at)
print("Error:", abs(intensity_pt-ref_intensity).max()/ref_intensity.max())

## Cython implementation

In [ ]:
%reload_ext Cython

In [ ]:
%%cython --compile-args=-fopenmp --link-args=-fopenmp
import numpy as np
import math
from cython.parallel import prange
import cython

from libc.math cimport sin
#cdef extern from "math.h" nogil:
#    double sin(double x)
    
cdef extern from "complex.h" nogil:
    double cabs(double complex)

@cython.wraparound(False)
@cython.boundscheck(False)
@cython.cdivision(True)
def Debye_cython(double[::1] Q, double[::1] r, long N, double complex[::1] f_at):
    cdef:
        double tmp
        long i_r, i_Q, size_r, size_Q
        double[::1] res
    
    size_r = r.size
    size_Q = Q.size
    res = np.zeros((size_Q))
    for i_Q in prange(size_Q, nogil=True):
        tmp = 0.0
        for i_r in range(size_r):
            tmp = tmp + sin(Q[i_Q]*r[i_r])/(Q[i_Q]*r[i_r])
        res[i_Q] = (N + 2*tmp)*cabs(f_at[i_Q])**2
    return res

In [ ]:
%timeit Debye_cython(Q,r_ij_pt,N,f_at)
intensity_ct = Debye_cython(Q,r_ij_pt,N,f_at)
print("Error:", abs(intensity_ct-ref_intensity).max()/ref_intensity.max())